# Лаба 4

## Анализ датасета

Сначала соберу пути картинок и вытащу цвет из имени файла (там всё размечено через разделитель `$$`).Отрежу совсем редкие цвета. Потом сделаю общий тест и из оставшегося пула сниму подвыборку в 2000 примеров со стратификацией

In [9]:
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("device:", device)

device: mps


в basename после сплита по `$$` на четвёртой позиции лежит строка цвета (`Black`, `Silver`, и т.д.).

In [3]:
FSEP = "$" * 2
DATA_ROOT = Path.cwd()
if not (DATA_ROOT / "Volkswagen").is_dir():
    DATA_ROOT = Path("/lab_4")
assert DATA_ROOT.is_dir(), DATA_ROOT

def iter_image_paths(root: Path):
    exts = {".jpg", ".jpeg", ".png"}
    for dp, _, fns in os.walk(root):
        if ".ipynb_checkpoints" in dp:
            continue
        for fn in fns:
            if Path(fn).suffix.lower() in exts:
                yield Path(dp) / fn

def parse_color(path: Path):
    parts = path.name.split(FSEP)
    if len(parts) < 4:
        return None
    return parts[3]
paths = []
labels_raw = []
for p in iter_image_paths(DATA_ROOT):
    c = parse_color(p)
    if c is None:
        continue
    paths.append(p)
    labels_raw.append(c)




cnt = Counter(labels_raw)
MIN_PER_CLASS = 15
valid = {c for c, n in cnt.items() if n >= MIN_PER_CLASS}
paths, labels_raw = zip(
    *[(p, c) for p, c in zip(paths, labels_raw) if c in valid]
)
paths, labels_raw = list(paths), list(labels_raw)
cnt2 = Counter(labels_raw)
print("изображений:", len(paths), "классов:", len(cnt2))
print("частые:", cnt2.most_common(6))

изображений: 61807 классов: 19
частые: [('Black', 14317), ('Grey', 9474), ('White', 9395), ('Blue', 8483), ('Silver', 7770), ('Red', 6095)]




Сначала train_test_split со стратификацией на полном отфильтрованном наборе. Потом train_size=2000 внутри тренировочного пула.


In [4]:
classes = sorted(set(labels_raw))
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
y_all = np.array([class_to_idx[c] for c in labels_raw], dtype=np.int64)
idx_all = np.arange(len(paths))

idx_pool, idx_test, y_pool, y_test = train_test_split(
    idx_all,
    y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=SEED,
)

MAX_TRAIN = 2000
sub = StratifiedShuffleSplit(n_splits=1, train_size=MAX_TRAIN, random_state=SEED)
idx_train_rel, _ = next(sub.split(np.zeros(len(y_pool)), y_pool))
idx_train = idx_pool[idx_train_rel]
print("train:", len(idx_train), "test:", len(idx_test), "число классов:", len(classes))

train: 2000 test: 12362 число классов: 19


## Общий Dataset + аугментации

Для претрейненных сетей использую стандартную нормализацию ImageNet. Для модели с нуля  чуть более жёсткий random resized crop, чтобы маленькая сеть хоть как-то обобщала.

In [5]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)





class CarColorDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        p = self.paths[i]
        y = int(self.labels[i])
        img = Image.open(p).convert("RGB")
        return self.transform(img), y





def loaders_from_indices(idx_tr, idx_te, tfm_train, tfm_eval, batch_size=32, num_workers=0):
    ds_tr = CarColorDataset(
        [paths[i] for i in idx_tr], y_all[idx_tr], tfm_train
    )
    ds_te = CarColorDataset(
        [paths[i] for i in idx_te], y_all[idx_te], tfm_eval
    )
    return (
        DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=num_workers),
        DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=num_workers),
    )





tfm_eval = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

tfm_train_big = transforms.Compose(
    [
        transforms.RandomResizedCrop(224, scale=(0.65, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.15, 0.15, 0.15, 0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

tfm_train_scratch = transforms.Compose(
    [
        transforms.RandomResizedCrop(224, scale=(0.55, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.25, 0.25, 0.25, 0.08),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

In [6]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        ys.append(y.numpy())
        ps.append(pred)
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    return f1_score(y_true, y_pred, average="macro"), y_true, y_pred


def train_named(name, model, train_loader, test_loader, epochs, lr, weight_decay=0.01):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        n = 0
        for x, y in tqdm(train_loader, desc=f"{name} ep{ep}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = crit(logits, y)
            loss.backward()
            opt.step()
            running += loss.item() * x.size(0)
            n += x.size(0)
        f1m, _, _ = evaluate(model, test_loader)
        history.append((ep, running / max(1, n), f1m))
        print(f"[{name}] ep={ep} loss={running / max(1, n):.4f} F1_macro(test)={f1m:.4f}")
    return model, history

## Модель 1 ResNet-18 и веса ImageNet-1k

Беру не гигантский ResNet50, потому что на CPU в лабораторной это бы просто вечность,суть трансфера от этого не меняется заменяю голову на len(classess) и учу всю сеть с небольшим lr, потому что данных мало

In [7]:
n_cls = len(classes)
rn = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
rn.fc = nn.Linear(rn.fc.in_features, n_cls)
train_loader, test_loader = loaders_from_indices(
    idx_train, idx_test, tfm_train_big, tfm_eval, batch_size=32
)

rn, hist_rn = train_named(
    "ResNet18-IN1k", rn, train_loader, test_loader, epochs=4, lr=3e-4
)
f1_rn, yt, yp_rn = evaluate(rn, test_loader)
print("Итог ResNet18 F1_macro:", f1_rn)

[ResNet18-IN1k] ep=1 loss=1.2904 F1_macro(test)=0.2897


[ResNet18-IN1k] ep=2 loss=0.8279 F1_macro(test)=0.3391


[ResNet18-IN1k] ep=3 loss=0.6252 F1_macro(test)=0.3892


[ResNet18-IN1k] ep=4 loss=0.5247 F1_macro(test)=0.3039
Итог ResNet18 F1_macro: 0.30393158957198174


## Модель 2 ConvNeXt-Tiny, претрейн ImageNet-22k

вторая часть задания у ConvNeXt в timm есть чекпоинт fb_in22k,  он обучался на 22k классах, то есть это другой источник знаний,  не ту же самую тысячу классов только с другим именем файла. Голову заменяю на линейный классификатор по фичам.

In [10]:
cn = timm.create_model(
    "convnext_tiny.fb_in22k", pretrained=True, num_classes=n_cls
)


train_loader, test_loader = loaders_from_indices(
    idx_train, idx_test, tfm_train_big, tfm_eval, batch_size=24
)
cn, hist_cn = train_named(
    "ConvNeXt-tiny-IN22k", cn, train_loader, test_loader, epochs=10, lr=3e-4
)
f1_cn, _, yp_cn = evaluate(cn, test_loader)
print("Итог ConvNeXt F1_macro:", f1_cn)

[ConvNeXt-tiny-IN22k] ep=1 loss=1.2480 F1_macro(test)=0.3565


[ConvNeXt-tiny-IN22k] ep=2 loss=0.8079 F1_macro(test)=0.3542


[ConvNeXt-tiny-IN22k] ep=3 loss=0.5862 F1_macro(test)=0.3915


[ConvNeXt-tiny-IN22k] ep=4 loss=0.4200 F1_macro(test)=0.4709


[ConvNeXt-tiny-IN22k] ep=5 loss=0.3207 F1_macro(test)=0.4804


[ConvNeXt-tiny-IN22k] ep=6 loss=0.2286 F1_macro(test)=0.3852


[ConvNeXt-tiny-IN22k] ep=7 loss=0.1569 F1_macro(test)=0.4566


[ConvNeXt-tiny-IN22k] ep=8 loss=0.1614 F1_macro(test)=0.4554


[ConvNeXt-tiny-IN22k] ep=9 loss=0.1066 F1_macro(test)=0.4130


[ConvNeXt-tiny-IN22k] ep=10 loss=0.0939 F1_macro(test)=0.4916
Итог ConvNeXt F1_macro: 0.4916083131354982


## Модель 3 своя CNN

По заданию нужен классификатор без   претрейна при ~2000 обучающих снимках и нескольких десятках классов тяжёлая сеть переобучится, поэтому я специально беру относительно узкую архитектуру.
Несколько этапов свёрток с пулинго  м дают от локальных контрастов и текстур к более крупным паттернам, что хорошо стыкуется с тем, что цвет кузова на фронтальном кадре читается по локальным участкам. На каждом этапе   две свёртки 3×3 подряд*(как в VGG) . Каналы растут 32 → 256 , к свёрткам добавлены BatchNorm и ReLU, чтобы стабильнее учиться на малых батчах. Вместо огромного полносвязного слоя на развёрнутую карту признаков стоит AdaptiveAvgPool +   один Linear меньше параметров в классификаторе и ниже риск зазубрить точные позиции на сетке, аугментации для этой модели   жёстче, чем у претрейненных, потому что   готовых весов ImageNet у неё нет.

In [11]:
class TinyColorCNN(nn.Module):
    def __init__(self, n_cls):
        super().__init__()

        def stage(cin, cout):

            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.backbone = nn.Sequential(
            stage(3, 32),
            stage(32, 64),
            stage(64, 128),
            stage(128, 256),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, n_cls),
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)


scratch = TinyColorCNN(n_cls)
train_loader, test_loader = loaders_from_indices(
    idx_train, idx_test, tfm_train_scratch, tfm_eval, batch_size=32
)

scratch, hist_sc = train_named(
    "TinyCNN-scratch", scratch, train_loader, test_loader, epochs=20, lr=8e-4
)
f1_sc, _, yp_sc = evaluate(scratch, test_loader)
print("Итог scratch F1_macro:", f1_sc)

[TinyCNN-scratch] ep=1 loss=1.7961 F1_macro(test)=0.1556


[TinyCNN-scratch] ep=2 loss=1.4873 F1_macro(test)=0.1657


[TinyCNN-scratch] ep=3 loss=1.4323 F1_macro(test)=0.1684


[TinyCNN-scratch] ep=4 loss=1.4355 F1_macro(test)=0.1913


[TinyCNN-scratch] ep=5 loss=1.3434 F1_macro(test)=0.2058


...

[TinyCNN-scratch] ep=16 loss=1.0935 F1_macro(test)=0.2358


[TinyCNN-scratch] ep=17 loss=1.1017 F1_macro(test)=0.2638


[TinyCNN-scratch] ep=18 loss=1.0522 F1_macro(test)=0.2373


[TinyCNN-scratch] ep=19 loss=1.0565 F1_macro(test)=0.2664


[TinyCNN-scratch] ep=20 loss=1.0389 F1_macro(test)=0.2755
Итог scratch F1_macro: 0.27554617379503604


## Сравнение и про F1 macro

Ниже простая таблица. F1 macro усредняет F1 по классам с равными весами, поэтому редкие цвета вносят такой же вклад, как частые. Если accuracy высокий, а macro низкий - модель, скорее всего,  сидит на белом/чёрном/серебристом и игнорит хвост. Если macro близок к micro/accuracy - классы отрабатываются более равномерно.

In [12]:
rows = [
    ("ResNet18 + ImageNet-1k", f1_rn),
    ("ConvNeXt-tiny + ImageNet-22k", f1_cn),
    ("TinyCNN с нуля", f1_sc),
]
for name, score in sorted(rows, key=lambda x: -x[1]):
    print(f"{score:.4f}  {name}")

best = max(rows, key=lambda x: x[1])
print("Лучше по F1_macro:", best[0], best[1])

print("Отчёт по лучшему из претрейнов (ResNet) — видно слабые классы:")
print(classification_report(yt, yp_rn, target_names=classes, zero_division=0))

0.4916  ConvNeXt-tiny + ImageNet-22k
0.3039  ResNet18 + ImageNet-1k
0.2755  TinyCNN с нуля
Лучше по F1_macro: ConvNeXt-tiny + ImageNet-22k 0.4916083131354982
Отчёт по лучшему из претрейнов (ResNet) — видно слабые классы:
              precision    recall  f1-score   support

       Beige       0.45      0.07      0.13       120
       Black       0.84      0.71      0.77      2864
        Blue       0.88      0.71      0.79      1697
      Bronze       0.00      0.00      0.00        66
       Brown       0.54      0.15      0.24       182
        Gold       0.56      0.12      0.19        43
       Green       0.23      0.22      0.23       156
        Grey       0.48      0.61      0.54      1895
      Maroon       0.00      0.00      0.00         5
 Multicolour       0.00      0.00      0.00        39
      Orange       0.21      0.58      0.30       112
        Pink       0.00      0.00      0.00        17
      Purple       0.33      0.01      0.03        72
         Red       0.9

## Выводы по тому,что получилось

F1 macro я смотрел именно потому, что классы по цвету у нас несбалансированные одна только accuracy могла бы создать иллюзию, что всё ок, пока модель угадывает чёрный, белый и серебристый, а редкие оттенки вообще не ловит. Macro усредняет F1 по классам поровну, поэтому слабые цвета так же бьют по метрике, как сильные и это хорошо видно.

По факту у меня вышло так ConvNeXt-tiny с претрейном на ImageNet-22k ~0.49 по macro, ResNet18 с ImageNet-1k ~0.30, своя CNN с нуля ~0.28.То есть трансфер в целом ожидаемо обгоняет обучение с нуля , но разница между двумя претрейнами тут вообще не косметическая: ConvNeXt ушёл заметно вперёд.Наверное, сочетание архитектуры и того, что веса видели кучу классов на IN-22k, лучше подошло под нашу задачу при тех же 2000 обучающих картинках.

Что означает macro около 0.49 у лучшей модели в среднем по классам модель ещё часто ошибается особенно там где мало примеров. По ResNet у массовых цветов F1 высокий, а у хвоста —низкий recall или вообще нули, и именно это тянет macro вниз. Для идеального отчёта логично было бы ещё распечатать classification_report именно для ConvNeXt, потому что он у меня лучший по macro, а не ResNet — просто ResNet нагляднее показал, как выглядит перекос.

Итого при малом train имеет смысл опираться на претрейн, и от источника претрейна или выбора бэкбона качество реально зависит сильнее, чем я изначально думал. Полностью задачу это не закрывает  на редких цветах всё ещё больно , но как сравнение трёх подходов картина довольно ясная.